In [1]:
print("hwsds")

hwsds


In [2]:
import numpy as np
import cv2
import base64

In [3]:
# 1. The Raw AI Output (Tiny 2x2 mask)
mask = np.array([
    [0, 1],
    [1, 0]
])

In [5]:
mask

array([[0, 1],
       [1, 0]])

In [4]:
# Multiply by 255 to make the water pure white
mask_visual = (mask * 255).astype(np.uint8)
print("1. Mask Visual Matrix:\n", mask_visual)

1. Mask Visual Matrix:
 [[  0 255]
 [255   0]]


In [6]:
# 2. OpenCV Encoding
success, buffer = cv2.imencode('.png', mask_visual)
print("\n2. Success Flag:", success)
print("3. Buffer (First 15 numbers):", buffer[:15])


2. Success Flag: True
3. Buffer (First 15 numbers): [137  80  78  71  13  10  26  10   0   0   0  13  73  72  68]


In [8]:
buffer

array([137,  80,  78,  71,  13,  10,  26,  10,   0,   0,   0,  13,  73,
        72,  68,  82,   0,   0,   0,   2,   0,   0,   0,   2,   8,   0,
         0,   0,   0,  87, 221,  82, 248,   0,   0,   0,  14,  73,  68,
        65,  84,   8,  29,  99, 100, 248, 207, 248, 159,  17,   0,   6,
        10,   2,   2, 157,  62, 129,  85,   0,   0,   0,   0,  73,  69,
        78,  68, 174,  66,  96, 130], dtype=uint8)

In [7]:
# 3. Base64 Translation
mask_base64 = base64.b64encode(buffer).decode('utf-8')
print("\n4. Final Base64 String:\n", mask_base64)


4. Final Base64 String:
 iVBORw0KGgoAAAANSUhEUgAAAAIAAAACCAAAAABX3VL4AAAADklEQVQIHWNk+M/4nxEABgoCAp0+gVUAAAAASUVORK5CYII=


In [ ]:
import streamlit as st
import folium
from folium.plugins import Draw, Geocoder, LocateControl
from streamlit_folium import st_folium
import requests
import base64
import numpy as np
import cv2
# 1. Page Configuration (Must be the very first Streamlit command)
st.set_page_config(
    page_title="AI Water Segmentation",
    page_icon="🌍",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ==========================================
# 🧭 SIDEBAR: INSTRUCTIONS & CONTACT INFO
# ==========================================
with st.sidebar:
    st.header("📖 How to Use")
    st.markdown("""
    1. **Search** for a location using the dropdown or the map's search icon.
    2. **Change Map View** using the layer icon (top right) to see satellite or street views.
    3. **Draw a Rectangle** over a body of water using the square icon on the left.
    4. **Run Analysis** to let the AI process the satellite data!
    """)
    
    # The Bounding Box Size Warning
    st.warning("""
    **📏 Bounding Box Guide:**
    - **Too Small (< 50m):** The satellite resolution (10m/pixel) cannot detect it.
    - **Too Big (> 10km):** Downloading massive satellite files will overload the network.
    - **Perfect Size:** Draw an area roughly 1 to 3 kilometers wide (the size of a large neighborhood).
    """, icon="⚠️")
    
    st.divider()
    
    # Author & Contact Info
    st.header("👨‍💻 About the Author")
    st.markdown("""
    **Tamer** *Full Stack AI Engineer* 📍 Cairo, Egypt  
    
    Built with Python, FastAPI, and a custom U-Net architecture to process live Sentinel-2 multispectral satellite data.
    """)
    
    st.markdown("### 📬 Support & Links")
    st.markdown("[📧 Email Support](tamer.elkot.ai@gmail.com)")
    st.markdown("[💼 LinkedIn](https://www.linkedin.com/in/tamer-elkot/)")
    st.markdown("[🐙 GitHub Repository](https://github.com/tamer-elkoT)")
    st.markdown("[📧 Portfolio](https://tamerelkot.netlify.app/)")
# 1. Page Configuration
st.set_page_config(page_title="🌍 Water Segmentation AI", layout="wide")
st.title("🌍 Multispectral Water Segmentation")
st.markdown("Select a location, draw a bounding box, and let the U-Net extract the water bodies.")

# 🚨 THE DROPDOWN SEARCH: Type 2 letters to filter this list!
locations = {
    "Aswan Low Dam, Egypt": [24.0350, 32.8750],
    "Nile Delta, Egypt": [31.4000, 31.0000],
    "Lake Nasser, Egypt": [22.7500, 32.7500],
    "Lake Victoria, Africa": [-1.0000, 33.0000],
    "Lake Mead, USA": [36.0163, -114.7370]
}
selected_loc = st.selectbox("🔍 Quick Jump to Location (Type to search):", list(locations.keys()))
map_center = locations[selected_loc]

# 2. Initialize the Map
# m = folium.Map(location=map_center, zoom_start=12, tiles="CartoDB positron")
# Google Map UI
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")
# Google Earth UI
# m = folium.Map(
#     location=map_center, 
#     zoom_start=12, 
#     tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
#     attr='Esri World Imagery'
# )
# Add Layer 1: Satellite View (Esri World Imagery)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellite View'
).add_to(m)

# Add Layer 2: Minimalist View (Best for making the AI blue mask pop!)
folium.TileLayer(
    tiles='CartoDB positron',
    name='Minimalist (AI View)'
).add_to(m)
#  Add the Layer Control button so the user can switch between them
folium.LayerControl().add_to(m)
Geocoder().add_to(m)
# Add the Locate Me GPS tracking button
LocateControl().add_to(m)
# Add the Drawing Tools
Draw(
    draw_options={'polyline': False, 'polygon': False, 'circle': False, 'marker': False, 'circlemarker': False, 'rectangle': True},
    edit_options={'edit': False}
).add_to(m)

# 4. Display the Map
st.subheader("Select Target Area")
map_output = st_folium(m, width=1000, height=500)

st.subheader("📡 AI Extraction")

# 5. Extract Coordinates and Connect to API
if map_output and map_output.get("last_active_drawing"):
    coordinates = map_output["last_active_drawing"]["geometry"]["coordinates"][0]
    lons = [p[0] for p in coordinates]
    lats = [p[1] for p in coordinates]
    bbox = [min(lons), min(lats), max(lons), max(lats)]
    
    st.success(f"The Selected Coordinates(BBox): `{bbox}`")
    
    # THE API CONNECTION BUTTON
    if st.button("Get the Segmented water body", type="primary"):
        
        # Streamlit spinner to show the user the AI is thinking
        with st.spinner("Downloading satellite data and running neural network...\n That will take few minutes"):
            try:
                # 1. Send the data to your FastAPI backend
                api_url = "http://127.0.0.1:8000/api/predict"
                response = requests.post(api_url, json={"bbox": bbox})
                
                if response.status_code == 200:
                    data = response.json()
                    
                    st.markdown("---")
                    st.subheader("📊 Analytics Report")
                    
                    # 2. Display the stats in beautiful UI metric cards
                    col1, col2, col3, col4 = st.columns(4)
                    col1.metric("Capture Date", data["capture_date"])
                    col2.metric("Total Scan Area", f"{data['total_area_km^2']} km²")
                    col3.metric("Water Area", f"{data['water_area_km^2']} km²")
                    col4.metric("Water Coverage", f"{data['water_percentage']}%")
                    
                    # 3. THE UNBOXING: Base64 back to Image
                    img_bytes = base64.b64decode(data["mask_base64"])
                    mask_array = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_UNCHANGED)
                    
                    # 4. The Computer Vision Trick: Turn White pixels into Blue water pixels!
                    blue_mask = np.zeros((128, 128, 3), dtype=np.uint8)
                    blue_mask[mask_array == 255] = [0, 150, 255] # RGB color for water
                    
                    # Display the final image
                    st.image(blue_mask, caption="AI Predicted Water Mask", width=400)
                    
                else:
                    st.error(f"Backend Error: {response.text}")
                    
            except Exception as e:
                st.error(f"Could not connect to FastAPI. Is your server running? Error: {e}")
else:
    st.info("👈 Please draw a rectangle on the map to begin.", icon="ℹ️")